# Steel Heat Treatment — Mechanical Property Prediction
### XGBoost Multi-Output Regression · FYP Final Model (v2)

**Dataset**: `steel_heat_treatment.csv`  — 570 samples, 4 heat treatment processes
**Targets** : Tensile Strength (MPa) · Yield Strength (MPa) · Hardness (HB) · Elongation (%) · Fatigue Strength (MPa)
**Features**: Chemical composition (9 elements) + Process type + HT temperature + Soaking time + Cooling medium + Tempering params


In [ ]:
import warnings; warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.multioutput import MultiOutputRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

from xgboost import XGBRegressor
import shap
import joblib, json, os

plt.rcParams.update({
    "figure.facecolor": "#0e1117", "axes.facecolor": "#1a1d2e",
    "axes.edgecolor": "#334",      "axes.labelcolor": "#c8d8ff",
    "xtick.color": "#8a9bc0",      "ytick.color": "#8a9bc0",
    "text.color": "#c8d8ff",       "grid.color": "#2a3050",
    "grid.linestyle": "--",        "grid.alpha": 0.5,
    "font.family": "DejaVu Sans",  "figure.dpi": 110,
})
COLORS = ["#4c72b0", "#dd8452", "#55a868", "#c44e52", "#8172b3"]
print("Imports OK")


## 1  Load Dataset

In [ ]:
df = pd.read_csv("steel_heat_treatment.csv")
print(f"Shape: {df.shape}")
print(f"\nProcess counts:")
print(df["Process"].value_counts())
print(f"\nCooling medium counts:")
print(df["Cooling_Medium"].value_counts())
df.head()


## 2  Exploratory Data Analysis

In [ ]:
targets = ["Tensile_MPa","Yield_MPa","Hardness_HB","Elongation_pct","Fatigue_MPa"]
print("Target property statistics:")
df[targets].describe().round(1)


In [ ]:
# Property distributions by process type
process_colors = {"Quench_Temper":"#4c72b0","Normalizing":"#55a868",
                  "Annealing":"#dd8452","Stress_Relief":"#c44e52"}
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
axes = axes.flatten()
for ax, tgt in zip(axes, targets):
    for proc, grp in df.groupby("Process"):
        ax.hist(grp[tgt], bins=28, alpha=0.55, label=proc,
                color=process_colors.get(proc, "#888"), edgecolor="none")
    ax.set_xlabel(tgt); ax.set_ylabel("Count")
    ax.set_title(f"Distribution of {tgt}", fontsize=9)
    ax.legend(fontsize=7)
axes[-1].set_visible(False)
plt.suptitle("Mechanical Property Distributions by Process", fontsize=12)
plt.tight_layout(); plt.show()


In [ ]:
# Box plots — property vs process
fig, axes = plt.subplots(1, len(targets), figsize=(18, 5))
proc_list = list(process_colors.keys())
for ax, tgt, col in zip(axes, targets, COLORS):
    data = [df[df["Process"]==p][tgt].values for p in proc_list]
    bplot = ax.boxplot(data, patch_artist=True,
                       medianprops=dict(color="white", linewidth=2))
    for patch, pc in zip(bplot["boxes"], process_colors.values()):
        patch.set_facecolor(pc + "88")
    ax.set_xticks(range(1, len(proc_list)+1))
    ax.set_xticklabels(proc_list, rotation=20, ha="right", fontsize=7)
    ax.set_ylabel(tgt, fontsize=9); ax.set_title(tgt, fontsize=9)
plt.suptitle("Properties by Process Type", fontsize=12, y=1.02)
plt.tight_layout(); plt.show()


In [ ]:
# Correlation heatmap (chemistry + process params vs targets)
num_cols = ["C","Si","Mn","Ni","Cr","Mo","HT_Temp_C","Soaking_Time_min",
            "Tempering_Temp_C","Tempering_Time_min"] + targets
corr = df[num_cols].corr()
fig, ax = plt.subplots(figsize=(13, 9))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, ax=ax, cmap="coolwarm", center=0,
            annot=True, fmt=".2f", annot_kws={"size": 7},
            linewidths=0.3, linecolor="#1a1d2e",
            cbar_kws={"shrink": 0.7})
ax.set_title("Correlation Heatmap", fontsize=12, pad=10)
plt.tight_layout(); plt.show()


## 3  Feature Engineering

Physically meaningful derived features improve model accuracy.

In [ ]:
def carbon_equiv(row):
    """IIW carbon equivalent — hardenability proxy."""
    return (row.C + row.Mn/6 + row.Si/24
            + row.Ni/40 + row.Cr/5 + row.Mo/4 + row.Cu/15)

def a3_temp_row(row):
    """Andrews (1965): A3 line in °C."""
    return (912 - 203*np.sqrt(max(row.C, 1e-6))
            - 30*row.Mn + 44.7*row.Si - 15.2*row.Ni + 31.5*row.Mo)

def hollomon_jaffe_row(row):
    """H = T(K) × (18 + log10(t_hours)); 0 for non-Q&T processes."""
    if row.Tempering_Temp_C <= 0:
        return 0.0
    t_h = max(row.Tempering_Time_min / 60.0, 0.001)
    return (row.Tempering_Temp_C + 273.15) * (18.0 + np.log10(t_h))

df["Carbon_Equiv"]    = df.apply(carbon_equiv, axis=1)
df["A3_Temp_C"]       = df.apply(a3_temp_row, axis=1)
df["Delta_HT_A3"]     = df["HT_Temp_C"] - df["A3_Temp_C"]   # >0 = fully austenitized
df["Hollomon_Jaffe"]  = df.apply(hollomon_jaffe_row, axis=1)
df["C_x_Cr"]          = df["C"] * df["Cr"]                   # carbide-forming interaction

print("Engineered features summary:")
eng_cols = ["Carbon_Equiv","A3_Temp_C","Delta_HT_A3","Hollomon_Jaffe","C_x_Cr"]
print(df[eng_cols].describe().round(3).to_string())


## 4  One-Hot Encoding & Train/Test Split

In [ ]:
# One-hot encode Process (4 dummies) and Cooling_Medium (up to 6 dummies)
df_enc = pd.get_dummies(df, columns=["Process", "Cooling_Medium"], drop_first=False)

FEATURE_COLS = [
    # chemistry (9)
    "C", "Si", "Mn", "P", "S", "Ni", "Cr", "Cu", "Mo",
    # engineered (5)
    "Carbon_Equiv", "A3_Temp_C", "Delta_HT_A3", "Hollomon_Jaffe", "C_x_Cr",
    # process parameters (4)
    "HT_Temp_C", "Soaking_Time_min", "Tempering_Temp_C", "Tempering_Time_min",
    # process dummies (4)
    "Process_Quench_Temper", "Process_Normalizing",
    "Process_Annealing", "Process_Stress_Relief",
    # cooling medium dummies (up to 6)
    "Cooling_Medium_Water", "Cooling_Medium_Oil", "Cooling_Medium_Polymer",
    "Cooling_Medium_Air", "Cooling_Medium_Furnace", "Cooling_Medium_Salt Bath",
]
# Keep only columns actually present
FEATURE_COLS = [c for c in FEATURE_COLS if c in df_enc.columns]
print(f"Total features: {len(FEATURE_COLS)}")
print(", ".join(FEATURE_COLS))

TARGET_COLS = ["Tensile_MPa","Yield_MPa","Hardness_HB","Elongation_pct","Fatigue_MPa"]

X = df_enc[FEATURE_COLS].astype(float)
y = df_enc[TARGET_COLS].astype(float)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42)
print(f"\nTrain: {X_train.shape}   Test: {X_test.shape}")


## 5  Model Training — GridSearchCV

In [ ]:
base_xgb = XGBRegressor(
    objective="reg:squarederror",
    tree_method="hist",
    random_state=42,
    n_jobs=-1,
)

param_grid = {
    "estimator__n_estimators":    [200, 400],
    "estimator__max_depth":       [4, 6],
    "estimator__learning_rate":   [0.05, 0.10],
    "estimator__subsample":       [0.80, 1.00],
    "estimator__colsample_bytree":[0.70, 0.90],
    "estimator__min_child_weight":[3, 5],
}

model = MultiOutputRegressor(base_xgb)
gs = GridSearchCV(model, param_grid, cv=5, scoring="r2", n_jobs=-1, verbose=1)

print("Running GridSearchCV… (may take 5-10 minutes)")
gs.fit(X_train, y_train)

print(f"\nBest mean CV R² : {gs.best_score_:.4f}")
print(f"Best params     :")
for k, v in gs.best_params_.items():
    print(f"  {k}: {v}")

best_model = gs.best_estimator_


## 6  Model Evaluation

In [ ]:
y_pred = pd.DataFrame(
    best_model.predict(X_test),
    columns=TARGET_COLS, index=X_test.index
)

eval_rows = []
for t in TARGET_COLS:
    r2   = r2_score(y_test[t], y_pred[t])
    mae  = mean_absolute_error(y_test[t], y_pred[t])
    rmse = float(np.sqrt(mean_squared_error(y_test[t], y_pred[t])))
    mape = float(np.mean(np.abs((y_test[t] - y_pred[t]) / y_test[t].clip(lower=1))) * 100)
    eval_rows.append({"Target": t, "R²": round(r2,4),
                      "MAE": round(mae,2), "RMSE": round(rmse,2),
                      "MAPE (%)": round(mape,2)})

eval_df = pd.DataFrame(eval_rows).set_index("Target")
print(eval_df.to_string())
eval_df


In [ ]:
# Predicted vs Actual — 5 panels
fig, axes = plt.subplots(1, 5, figsize=(20, 4))
for ax, tgt, col in zip(axes, TARGET_COLS, COLORS):
    ax.scatter(y_test[tgt], y_pred[tgt], alpha=0.50, s=16,
               color=col, edgecolors="none")
    lo = min(y_test[tgt].min(), y_pred[tgt].min())
    hi = max(y_test[tgt].max(), y_pred[tgt].max())
    ax.plot([lo, hi], [lo, hi], "w--", lw=1, alpha=0.5)
    r2 = r2_score(y_test[tgt], y_pred[tgt])
    ax.set_title(f"{tgt}\nR² = {r2:.4f}", fontsize=9)
    ax.set_xlabel("Actual"); ax.set_ylabel("Predicted")
plt.suptitle("Predicted vs Actual — Test Set", fontsize=12)
plt.tight_layout(); plt.show()


In [ ]:
# Residuals distribution
fig, axes = plt.subplots(1, 5, figsize=(20, 4))
for ax, tgt, col in zip(axes, TARGET_COLS, COLORS):
    res = y_pred[tgt] - y_test[tgt]
    ax.hist(res, bins=28, color=col + "aa", edgecolor="none")
    ax.axvline(0, color="white", lw=1, linestyle="--")
    ax.set_title(f"{tgt}", fontsize=9)
    ax.set_xlabel("Predicted − Actual")
plt.suptitle("Residual Distributions — Test Set", fontsize=12)
plt.tight_layout(); plt.show()


## 7  Feature Importance

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(22, 6))
for ax, tgt, est, col in zip(axes, TARGET_COLS, best_model.estimators_, COLORS):
    imp = pd.Series(est.feature_importances_, index=FEATURE_COLS).nlargest(12)
    imp.sort_values().plot.barh(ax=ax, color=col + "cc")
    ax.set_title(tgt, fontsize=10)
    ax.set_xlabel("Importance")
    ax.tick_params(labelsize=7)
plt.suptitle("XGBoost Feature Importance — Top 12 per Target", fontsize=12)
plt.tight_layout(); plt.show()


## 8  SHAP Analysis

In [ ]:
# Compute SHAP values for each per-target estimator
shap_values_all = []
for est in best_model.estimators_:
    explainer = shap.TreeExplainer(est)
    sv = explainer.shap_values(X_test)
    shap_values_all.append(sv)
print("SHAP values computed.")


In [ ]:
# SHAP beeswarm summary — one per target
for sv, tgt in zip(shap_values_all, TARGET_COLS):
    shap.summary_plot(sv, X_test, feature_names=FEATURE_COLS,
                      max_display=14, show=False, plot_size=(9, 5))
    plt.title(f"SHAP Summary — {tgt}", fontsize=11)
    plt.tight_layout(); plt.show()


In [ ]:
# Global SHAP bar chart (mean |SHAP| averaged across all 5 targets)
mean_abs_shap = np.mean([np.abs(sv).mean(axis=0) for sv in shap_values_all], axis=0)
global_shap = pd.Series(mean_abs_shap, index=FEATURE_COLS).nlargest(15).sort_values()

fig, ax = plt.subplots(figsize=(8, 5))
global_shap.plot.barh(ax=ax, color="#4c72b0cc")
ax.set_xlabel("Mean |SHAP value| (averaged over all targets)")
ax.set_title("Global Feature Importance via SHAP", fontsize=11)
plt.tight_layout(); plt.show()


## 9  Partial Dependence Plots (key features)

In [ ]:
from sklearn.inspection import PartialDependenceDisplay

pdp_features = ["C", "Carbon_Equiv", "Hollomon_Jaffe", "HT_Temp_C", "Tempering_Temp_C"]
# keep only features that exist
pdp_features = [f for f in pdp_features if f in X_train.columns]

for est, tgt in zip(best_model.estimators_, TARGET_COLS):
    fig, axes = plt.subplots(1, len(pdp_features), figsize=(15, 3))
    PartialDependenceDisplay.from_estimator(
        est, X_train, features=pdp_features, ax=axes,
        line_kw={"color": "#4c72b0", "linewidth": 2.0},
    )
    fig.suptitle(f"Partial Dependence — {tgt}", fontsize=11, y=1.04)
    plt.tight_layout(); plt.show()


## 10  Save Model & Metadata

In [ ]:
os.makedirs("models", exist_ok=True)

# Per-target XGBoost JSON files (loaded by the Streamlit app)
for est, tgt in zip(best_model.estimators_, TARGET_COLS):
    fname = f"models/xgb_{tgt.lower().replace(' ','_')}.json"
    est.save_model(fname)
    print(f"Saved  {fname}")

# Full pipeline pickle (fallback)
joblib.dump(best_model, "models/best_model.pkl")
print("Saved  models/best_model.pkl")

# Metadata JSON (feature list, target list, training ranges)
training_ranges = {
    col: {"min": float(X_train[col].min()), "max": float(X_train[col].max())}
    for col in FEATURE_COLS
}
metadata = {
    "features":        FEATURE_COLS,
    "targets":         TARGET_COLS,
    "training_ranges": training_ranges,
    "model_metrics":   eval_df.reset_index().to_dict(orient="records"),
    "n_train":         int(len(X_train)),
    "n_test":          int(len(X_test)),
}
with open("models/model_metrics.json", "w") as f:
    json.dump(metadata, f, indent=2)
print("Saved  models/model_metrics.json")


## 11  Example Prediction

**AISI 4140 analogue** — Cr-Mo alloy steel, Quench & Temper

In [ ]:
# Build input row for AISI 4140-type steel
# Composition: 0.40C, 0.25Si, 0.85Mn, 1.05Cr, 0.20Mo, 0.20Ni
ex = {
    "C":0.40,"Si":0.25,"Mn":0.85,"P":0.010,"S":0.010,
    "Ni":0.20,"Cr":1.05,"Cu":0.10,"Mo":0.20,
    "HT_Temp_C":850.0,"Soaking_Time_min":45.0,
    "Tempering_Temp_C":550.0,"Tempering_Time_min":120.0,
    # process: Quench_Temper
    "Process_Quench_Temper":1,"Process_Normalizing":0,
    "Process_Annealing":0,"Process_Stress_Relief":0,
    # cooling: Oil
    "Cooling_Medium_Oil":1,"Cooling_Medium_Water":0,
    "Cooling_Medium_Polymer":0,"Cooling_Medium_Air":0,
    "Cooling_Medium_Furnace":0,"Cooling_Medium_Salt Bath":0,
}
# Engineered features
c,mn,si,ni,cr,mo,cu = ex["C"],ex["Mn"],ex["Si"],ex["Ni"],ex["Cr"],ex["Mo"],ex["Cu"]
ex["Carbon_Equiv"]   = c + mn/6 + si/24 + ni/40 + cr/5 + mo/4 + cu/15
ex["A3_Temp_C"]      = 912 - 203*np.sqrt(c) - 30*mn + 44.7*si - 15.2*ni + 31.5*mo
ex["Delta_HT_A3"]    = ex["HT_Temp_C"] - ex["A3_Temp_C"]
t_h                  = ex["Tempering_Time_min"] / 60.0
ex["Hollomon_Jaffe"] = (ex["Tempering_Temp_C"] + 273.15) * (18 + np.log10(t_h))
ex["C_x_Cr"]         = c * cr

X_ex = pd.DataFrame([ex])[FEATURE_COLS]
preds = best_model.predict(X_ex)[0]

print("Prediction  —  AISI 4140  (Q&T: 850°C austenitize / oil quench / 550°C×120 min temper)")
print("-" * 65)
for tgt, val in zip(TARGET_COLS, preds):
    print(f"  {tgt:<20s}:  {val:>8.1f}")

# Change tempering temperature to show effect
print("\nEffect of tempering temperature on Tensile Strength:")
for t_temp in [200, 300, 400, 500, 600]:
    ex2 = ex.copy()
    ex2["Tempering_Temp_C"]  = float(t_temp)
    t_h2 = ex2["Tempering_Time_min"] / 60.0
    ex2["Hollomon_Jaffe"] = (t_temp + 273.15) * (18 + np.log10(t_h2))
    X2 = pd.DataFrame([ex2])[FEATURE_COLS]
    p2 = best_model.predict(X2)[0]
    print(f"  T_temper = {t_temp:>4d}°C  →  "
          f"Tensile={p2[0]:>7.1f} MPa  HB={p2[2]:>6.1f}  Elong={p2[3]:>5.2f}%")
